In [ ]:
### ==========================================================================
### Two-Tower Retrieval Model for MovieLens 100K
### ==========================================================================
###
### This script builds a movie recommendation system using PyTorch.
###
### The idea: we learn a vector (embedding) for every user and every movie.
### If a user likes a movie, their vectors should point in the same direction
### (high dot product). If not, they should point away (low dot product).
###
### Architecture ("Two Towers"):
###   Tower 1 (User)  : user_id  -->  [Embedding table]  -->  32-dim vector
###   Tower 2 (Movie) : movie_id -->  [Embedding table]  -->  32-dim vector
###   Score = dot_product(user_vector, movie_vector)
###
### Loss: BPR (Bayesian Personalized Ranking)
###   For each (user, watched_movie) pair, we sample a random movie the user
###   has NOT watched. We then push the score of the watched movie higher
###   than the score of the unwatched movie.
###
### To recommend: compute user_vector dot ALL movie_vectors, pick the highest.
### ==========================================================================

# ── Install (only needed on Colab) ───────────────────────────────────────
# PyTorch and pandas come pre-installed on Colab, so nothing to install!

# ── Imports ──────────────────────────────────────────────────────────────


### 1. Imports and Hyperparameters

In [ ]:
import os                                   # For file/folder operations
import zipfile                              # To unzip the downloaded dataset
import urllib.request                       # To download the dataset from the web
import random                               # For sampling random negative movies

import numpy as np                          # Numerical arrays
import pandas as pd                         # DataFrames for loading CSV data
import torch                                # PyTorch: the deep learning framework
import torch.nn as nn                       # Neural network building blocks
import torch.nn.functional as F             # Useful functions like logsigmoid
from torch.utils.data import Dataset, DataLoader  # For batching training data

# ── Hyperparameters ──────────────────────────────────────────────────────

SEED   = 42       # Random seed so results are reproducible every run
DIM    = 32       # Size of each embedding vector (higher = more expressive but slower)
BATCH  = 2048     # How many (user, movie) pairs per training step
EPOCHS = 10       # How many full passes through the training data
LR     = 1e-2     # Learning rate: how big each optimization step is (0.01)

# Set all random seeds so results are the same every time
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# Use GPU if available (much faster), otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cpu


### 2. Load the Data

In [ ]:
# URL where the MovieLens 100K dataset lives
url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"

# Download and unzip only if we haven't already
if not os.path.exists("ml-100k"):
    print("Downloading MovieLens 100K …")
    urllib.request.urlretrieve(url, "ml-100k.zip")       # Download the zip file
    with zipfile.ZipFile("ml-100k.zip") as z:
        z.extractall(".")                                # Unzip into current folder
    print("Done.")

# Load the ratings file: each row is (user_id, movie_id, rating, timestamp)
ratings = pd.read_csv(
    "ml-100k/u.data", sep="\t",
    names=["user", "movie", "rating", "ts"],
)

# Load the movie titles file: each row maps movie_id -> movie title string
titles = pd.read_csv(
    "ml-100k/u.item", sep="|", encoding="latin-1",
    header=None, usecols=[0, 1],
    names=["movie", "title"],
)

# Merge so each rating row also has the movie's title
ratings = ratings.merge(titles, on="movie")

Done.


### 3. Build Vocabularies

In [ ]:
# Neural networks need numbers, not strings. So we map each unique user and
# movie title to a contiguous integer index (0, 1, 2, ...).

users  = sorted(ratings["user"].unique())    # List of all unique user IDs
movies = sorted(ratings["title"].unique())   # List of all unique movie titles

u2i = {u: i for i, u in enumerate(users)}   # user_id  -> integer index
m2i = {m: i for i, m in enumerate(movies)}  # title    -> integer index
i2m = {i: m for m, i in m2i.items()}         # integer  -> title (reverse lookup)

N_USERS  = len(users)    # Total number of unique users  (943)
N_MOVIES = len(movies)   # Total number of unique movies (1664)

# Build a set of movies each user has watched.
# We need this later to sample "negative" movies the user hasn't seen.
user_pos = {}                                           # dict: user_index -> set of movie_indices
for _, row in ratings.iterrows():                       # Loop through every rating
    uid = u2i[row["user"]]                              # Convert user ID to index
    mid = m2i[row["title"]]                             # Convert movie title to index
    user_pos.setdefault(uid, set()).add(mid)             # Add movie to this user's watched set

### 4. Train / Test Split

In [ ]:
# Randomly shuffle all 100k ratings, use the first 80k for training
# and the remaining 20k for testing.

perm     = np.random.permutation(len(ratings))                  # Random order
train_df = ratings.iloc[perm[:80_000]].reset_index(drop=True)   # First 80k = train
test_df  = ratings.iloc[perm[80_000:]].reset_index(drop=True)   # Last  20k = test

print(f"Users: {N_USERS}  Movies: {N_MOVIES}  Train: {len(train_df)}  Test: {len(test_df)}")

Users: 943  Movies: 1664  Train: 80000  Test: 20000


### 5. Dataset with Negative Sampling

In [ ]:
# For each real (user, watched_movie) pair, we also sample a random movie
# the user has NOT watched. This gives the model a "positive" and "negative"
# example to learn from.

class BPRDataset(Dataset):
    def __init__(self, df):
        # Pre-compute all (user_index, movie_index) pairs from the dataframe
        self.pairs = [(u2i[r["user"]], m2i[r["title"]]) for _, r in df.iterrows()]

    def __len__(self):
        return len(self.pairs)                           # Total number of training examples

    def __getitem__(self, i):
        u, pos = self.pairs[i]                           # Get the i-th (user, positive_movie) pair

        neg = random.randint(0, N_MOVIES - 1)            # Pick a random movie index
        while neg in user_pos[u]:                        # If the user already watched it...
            neg = random.randint(0, N_MOVIES - 1)        # ...pick again until we find one they haven't

        return u, pos, neg                               # Return (user, watched_movie, unwatched_movie)

# Create a DataLoader that shuffles and batches the training data
train_loader = DataLoader(
    BPRDataset(train_df),       # Our dataset
    batch_size=BATCH,           # 2048 samples per batch
    shuffle=True,               # Shuffle every epoch for better training
    drop_last=True,             # Drop incomplete last batch for consistent batch sizes
)

# Pre-compute test set as simple lists (no negative sampling needed for evaluation)
test_users  = [u2i[r["user"]]  for _, r in test_df.iterrows()]   # Test user indices
test_movies = [m2i[r["title"]] for _, r in test_df.iterrows()]   # Test movie indices (ground truth)

### 6. The Two-Tower Model

In [ ]:
# Two embedding tables: one for users, one for movies.
# Each table maps an integer index to a learned 32-dimensional vector.

class TwoTower(nn.Module):
    def __init__(self, n_users, n_movies, dim):
        super().__init__()
        self.user_emb  = nn.Embedding(n_users, dim)       # User  tower: n_users  x 32 table
        self.movie_emb = nn.Embedding(n_movies, dim)       # Movie tower: n_movies x 32 table

        # Xavier initialization spreads initial values evenly — helps training start well
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.movie_emb.weight)

# Create the model and move it to GPU (if available)
model = TwoTower(N_USERS, N_MOVIES, DIM).to(device)

# Adam optimizer: adjusts model weights to minimize the loss
opt = torch.optim.Adam(model.parameters(), lr=LR)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}\n")

Parameters: 83,424



### 7. Evaluation Function

In [ ]:
# After each epoch, we check: for each test (user, movie) pair, is the
# true movie in the model's top-K recommendations for that user?
# "top_100 = 0.40" means the true movie was in the top 100 picks 40% of the time.

@torch.no_grad()                             # No need to compute gradients during evaluation
def evaluate(ks=(1, 5, 10, 50, 100)):
    model.eval()                             # Put model in evaluation mode

    all_m = model.movie_emb.weight           # All movie embeddings: shape (N_MOVIES, 32)
    hits  = {k: 0 for k in ks}              # Count how many times true movie is in top-K
    total = len(test_users)                  # Total test examples

    # Convert test data to tensors on the right device
    t_users  = torch.tensor(test_users,  device=device)   # All test user indices
    t_movies = torch.tensor(test_movies, device=device)   # All test ground-truth movie indices

    # Process in chunks of 4096 to avoid running out of memory
    for start in range(0, total, 4096):
        end = min(start + 4096, total)

        u_emb  = model.user_emb(t_users[start:end])       # Look up user embeddings for this chunk
        scores = u_emb @ all_m.T                           # Dot product with ALL movies → (chunk, N_MOVIES)
        true_m = t_movies[start:end]                       # The actual movies these users watched

        for k in ks:
            topk = scores.topk(k, dim=1).indices           # Get indices of top-K highest-scored movies
            # Check if the true movie appears anywhere in the top-K list
            hits[k] += (topk == true_m.unsqueeze(1)).any(1).sum().item()

    # Return accuracy as a fraction (hits / total) for each K
    return {k: hits[k] / total for k in ks}

### 8. Training Loop

In [ ]:
# Each epoch:
#   1. Loop through batches of (user, positive_movie, negative_movie)
#   2. Compute BPR loss: push score(user, positive) above score(user, negative)
#   3. Update weights via backpropagation
#   4. Evaluate top-K accuracy on the test set

# Print a header row for the results table
header = f"{'Ep':>3}  {'Loss':>8}  {'top1':>7}  {'top5':>7}  {'top10':>7}  {'top50':>7}  {'top100':>7}"
print(header)
print("-" * len(header))

for ep in range(1, EPOCHS + 1):
    model.train()                                      # Put model in training mode
    total_loss, steps = 0.0, 0                         # Track loss for this epoch

    for uids, pos, neg in train_loader:                # Each batch: user indices, positive, negative
        uids = uids.to(device)                         # Move to GPU if available
        pos  = pos.to(device)
        neg  = neg.to(device)

        u_emb = model.user_emb(uids)                   # Look up user embeddings:    (BATCH, 32)
        p_emb = model.movie_emb(pos)                    # Look up positive movie emb: (BATCH, 32)
        n_emb = model.movie_emb(neg)                    # Look up negative movie emb: (BATCH, 32)

        pos_score = (u_emb * p_emb).sum(1)              # Dot product user·positive:  (BATCH,)
        neg_score = (u_emb * n_emb).sum(1)              # Dot product user·negative:  (BATCH,)

        # BPR loss: we want pos_score > neg_score.
        # logsigmoid(pos - neg) is high when pos >> neg, low when pos ≈ neg.
        # We negate it because we want to MINIMIZE loss (maximize the gap).
        loss = -F.logsigmoid(pos_score - neg_score).mean()

        opt.zero_grad()                                 # Clear old gradients
        loss.backward()                                 # Compute new gradients via backpropagation
        opt.step()                                      # Update model weights

        total_loss += loss.item()                       # Accumulate loss for logging
        steps += 1                                      # Count batches

    # Evaluate on test set after each epoch
    m = evaluate()
    print(f"{ep:>3}  {total_loss/steps:>8.4f}  "
          f"{m[1]:>7.4f}  {m[5]:>7.4f}  {m[10]:>7.4f}  {m[50]:>7.4f}  {m[100]:>7.4f}")

 Ep      Loss     top1     top5    top10    top50   top100
----------------------------------------------------------
  1    0.6455   0.0053   0.0229   0.0440   0.1729   0.2962
  2    0.3408   0.0057   0.0295   0.0544   0.2107   0.3432
  3    0.2527   0.0069   0.0313   0.0570   0.2250   0.3731
  4    0.2110   0.0070   0.0301   0.0573   0.2319   0.3866
  5    0.1903   0.0069   0.0307   0.0587   0.2317   0.3942
  6    0.1768   0.0072   0.0312   0.0596   0.2392   0.3992
  7    0.1588   0.0073   0.0306   0.0609   0.2441   0.4004
  8    0.1495   0.0069   0.0328   0.0614   0.2474   0.4019
  9    0.1409   0.0075   0.0336   0.0635   0.2454   0.4030
 10    0.1332   0.0073   0.0321   0.0622   0.2469   0.4056


### 9. Make Recommendations

In [ ]:
# Given a user ID, compute their embedding, dot-product it with every movie
# embedding, and return the top-K highest-scoring movies.

@torch.no_grad()                                        # No gradients needed for inference
def recommend(user_id, k=5):
    model.eval()                                        # Evaluation mode

    # Look up this user's learned embedding vector
    u = model.user_emb(torch.tensor([u2i[user_id]], device=device))   # Shape: (1, 32)

    # Score every movie: dot product of user vector with all movie vectors
    scores = (u @ model.movie_emb.weight.T).squeeze(0)                # Shape: (N_MOVIES,)

    # Pick the top-K highest scores
    topk = scores.topk(k)

    # Convert indices back to movie titles and return with scores
    return [(i2m[i.item()], s.item()) for i, s in zip(topk.indices, topk.values)]

# Show recommendations for two example users
print("\nTop 5 recommendations for user 42:")
for title, score in recommend(42):
    print(f"  score={score:+.3f}  {title}")

print("\nTop 5 recommendations for user 100:")
for title, score in recommend(100):
    print(f"  score={score:+.3f}  {title}")


Top 5 recommendations for user 42:
  score=+5.908  Raiders of the Lost Ark (1981)
  score=+5.752  Forrest Gump (1994)
  score=+5.667  Mrs. Doubtfire (1993)
  score=+5.552  E.T. the Extra-Terrestrial (1982)
  score=+5.490  Back to the Future (1985)

Top 5 recommendations for user 100:
  score=+7.009  G.I. Jane (1997)
  score=+6.845  Rainmaker, The (1997)
  score=+6.821  Seven Years in Tibet (1997)
  score=+6.727  Game, The (1997)
  score=+6.606  Air Force One (1997)
